# Reranking, Hybrid Retrieval, dan Guardrail

Lanjutan dari `02_embedding_indexing.ipynb`. Notebook ini dibuat standalone (menyalin ulang
parser dari notebook 01) supaya bisa dijalankan tanpa harus run notebook 01 dulu di kernel yang sama.

Isi notebook:
1. Setup model & koneksi ChromaDB
2. Parser bahan untuk input user
3. Skor efisiensi bahan (coverage_recipe + coverage_user, digabung pakai harmonic mean)
4. Hybrid retrieval (SBERT + keyword fallback, dengan tie-breaker efisiensi)
5. Reranking (Weighted Linear Combination)
6. Guardrail (cegah Llama3 "mengarang" resep) -- threshold T_jumlah, T_kategori, T_relevansi
7. Grid search bobot final (w1, w2)
8. Spot-check kualitatif dengan bobot final

In [1]:
import re
import html
import json
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb

c:\Users\Mayceren\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Setup: Model & Koneksi ChromaDB

In [2]:
MODEL_NAME = "LazarusNLP/all-indo-e5-small-v4"
CHROMA_DIR = "chroma_db"
COLLECTION_NAME = "resep_masakan"

model = SentenceTransformer(MODEL_NAME)
client = chromadb.PersistentClient(path=CHROMA_DIR)
collection = client.get_collection(COLLECTION_NAME)
print("Jumlah dokumen ter-index:", collection.count())


def add_query_prefix(text: str) -> str:
    return f"query: {text}"


def add_passage_prefix(text: str) -> str:
    return f"passage: {text}"

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8455.32it/s]


Jumlah dokumen ter-index: 5000


## 2. Parser Bahan untuk Input User

Sama seperti parser di `01_preprocessing.ipynb` (disalin ke sini). Idealnya nanti dipindah ke
satu modul `.py` bersama supaya tidak duplikat kalau ada perbaikan lagi.

In [3]:
UNITS = [
    "sendok makan",
    "sendok teh",
    "sdm",
    "sdt",
    "gram",
    "gr",
    "kg",
    "ons",
    "ml",
    "mili",
    "liter",
    "ltr",
    "l",
    "butir",
    "btr",
    "buah",
    "bh",
    "siung",
    "lembar",
    "lbr",
    "batang",
    "btg",
    "ikat",
    "ruas",
    "cm",
    "potong",
    "ptg",
    "biji",
    "bj",
    "ekor",
    "ekr",
    "genggam",
    "bungkus",
    "bks",
    "sachet",
    "sct",
    "papan",
    "buku",
    "helai",
    "hlai",
    "gelas",
    "gls",
    "cup",
    "mangkuk",
    "piring",
    "sisir",
    "kotak",
    "ktk",
    "kaleng",
    "klg",
    "botol",
    "btl",
]
UNIT_ALIASES = {
    "btr": "butir",
    "bh": "buah",
    "sdm": "sendok makan",
    "sdt": "sendok teh",
    "gr": "gram",
    "ltr": "liter",
    "l": "liter",
    "lbr": "lembar",
    "btg": "batang",
    "bks": "bungkus",
    "ptg": "potong",
    "bj": "biji",
    "ekr": "ekor",
    "gls": "gelas",
    "ktk": "kotak",
    "klg": "kaleng",
    "btl": "botol",
    "sct": "sachet",
    "hlai": "helai",
    "mili": "ml",
    "ml": "ml",
}
UNITS_SORTED = sorted(UNITS, key=len, reverse=True)
UNIT_PATTERN = "|".join(re.escape(u) for u in UNITS_SORTED)
QTY_PATTERN = (
    r"\d+\s*-\s*\d+"
    r"|\d+\s+\d+\s*/\s*\d+"
    r"|\d+\s*/\s*\d+"
    r"|\d+[.,]\d+"
    r"|\d+"
    r"|sesuai selera"
    r"|secukupnya"
    r"|sejumput"
    r"|beberapa"
)
# catatan: \b sengaja HANYA di belakang unit (bukan di depan) -- supaya qty+unit
# yang nempel tanpa spasi (mis. '100gr') tetap kebaca
INGREDIENT_REGEX = re.compile(
    rf"^\s*(?P<qty>{QTY_PATTERN})?\s*(?:(?P<unit>{UNIT_PATTERN})\b)?\.?\s*(?P<name>.+?)\s*$",
    re.IGNORECASE,
)


def clean_name(name: str) -> str:
    name = name.lower().strip()
    name = re.sub(r"\(.*?\)", "", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name


def parse_ingredient(raw: str) -> dict:
    raw = html.unescape(raw).strip()
    raw = re.sub(r"^[\-\*•]+\s*", "", raw)
    main, note = (raw.split(",", 1) + [""])[:2] if "," in raw else (raw, "")
    main = main.strip()
    m = INGREDIENT_REGEX.match(main)
    name = clean_name(m.group("name")) if (m and m.group("name")) else clean_name(main)
    return {"raw": raw, "name": name}


def parse_user_ingredients(text: str) -> list:
    items = [t.strip() for t in text.split(",") if t.strip()]
    parsed = [parse_ingredient(item) for item in items]
    return [p["name"] for p in parsed if p["name"]]


# test
print(parse_user_ingredients("2 butir telur, 100gr dada ayam, 2 siung bawang putih"))

['telur', 'dada ayam', 'bawang putih']


## 3. Skor Efisiensi Bahan

Definisi:
- **coverage_recipe** = bahan resep yang cocok / total bahan resep -- makin tinggi, makin sedikit bahan tambahan yang perlu dibeli
- **coverage_user** = bahan user yang cocok / total bahan yang diketik user -- makin tinggi, makin banyak bahan user yang benar-benar terpakai
- **skor_efisiensi_bahan** = harmonic mean dari keduanya

**PENTING (hasil debugging panjang):** kalau cuma pakai coverage_recipe saja, resep pendek yang cuma nyerempet 1 bahan bisa dapat skor sama tingginya dengan resep yang benar-benar memanfaatkan SEMUA bahan user, cuma karena total bahannya sedikit. Harmonic mean mencegah ini -- resep hanya dapat skor tinggi kalau KEDUA coverage sama-sama tinggi.

In [4]:
def ingredient_matches(u: str, r: str) -> bool:
    u, r = u.lower().strip(), r.lower().strip()
    if not u or not r:
        return False
    return u in r or r in u


def coverage_stats(user_ingredients: list, recipe_ingredients: list):
    """Return: (coverage_recipe, coverage_user, matched_items, missing_items)"""
    if not recipe_ingredients:
        return 0.0, 0.0, [], []
    matched_recipe_items, matched_user_items = [], set()
    for r_name in recipe_ingredients:
        hit = False
        for u_name in user_ingredients:
            if ingredient_matches(u_name, r_name):
                hit = True
                matched_user_items.add(u_name)
        if hit:
            matched_recipe_items.append(r_name)
    missing_items = [r for r in recipe_ingredients if r not in matched_recipe_items]
    coverage_recipe = len(matched_recipe_items) / len(recipe_ingredients)
    coverage_user = (
        len(matched_user_items) / len(user_ingredients) if user_ingredients else 0.0
    )
    return coverage_recipe, coverage_user, matched_recipe_items, missing_items


def harmonic_efficiency(cr: float, cu: float) -> float:
    if cr + cu == 0:
        return 0.0
    return (2 * cr * cu) / (cr + cu)


def fbeta_efficiency(cr: float, cu: float, beta: float = 2.0) -> float:
    """Alternatif harmonic mean berbobot -- beta>1 menekankan coverage_user lebih penting.
    Disimpan sebagai opsi eksperimen, tidak dipakai di formula final."""
    denom = (beta**2 * cr) + cu
    if denom == 0:
        return 0.0
    return (1 + beta**2) * (cr * cu) / denom


# test
cr, cu, matched, missing = coverage_stats(
    ["telur", "dada ayam", "bawang putih"],
    ["dada ayam", "telur", "garam", "merica", "kaldu jamur", "bawang putih"],
)
print(
    f"coverage_recipe={cr:.3f}, coverage_user={cu:.3f}, efisiensi={harmonic_efficiency(cr, cu):.3f}"
)
print(f"matched: {matched}, missing: {missing}")

coverage_recipe=0.500, coverage_user=1.000, efisiensi=0.667
matched: ['dada ayam', 'telur', 'bawang putih'], missing: ['garam', 'merica', 'kaldu jamur']


## 4. Hybrid Retrieval (SBERT + Keyword Fallback)

**Kenapa perlu hybrid:** pengujian awal menemukan bahwa retrieval SBERT murni pada query pendek
(2-3 bahan) bisa gagal total memasukkan resep yang relevan ke top-K, karena embedding cenderung
mengelompokkan berdasarkan pola topik dominan (mis. semua hasil "Telur Dadar" walau user
menyebut "dada ayam" juga). Solusinya: gabungkan hasil SBERT dengan pencarian kata kunci
langsung ke seluruh koleksi.

**PENTING (bug yang pernah ditemukan):** `keyword_candidates` HARUS diurutkan dengan
tie-breaker `coverage_recipe`, bukan cuma `matched_count` saja -- kalau tidak, banyak resep
yang seri jumlah match-nya (puluhan resep bisa seri), dan resep yang lebih efisien bisa
kepotong dari batas `max_keyword_extra` hanya karena urutan asli di dataset, bukan karena
kualitasnya.

In [5]:
def retrieve_candidates(user_text: str, top_k: int = 20) -> list:
    query_embedding = model.encode(
        [add_query_prefix(user_text)], normalize_embeddings=True, convert_to_numpy=True
    )
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )
    candidates = []
    for i in range(len(results["ids"][0])):
        meta = results["metadatas"][0][i]
        candidates.append(
            {
                "id": results["ids"][0][i],
                "title": meta["title"],
                "ingredient_names": json.loads(meta["ingredient_names"]),
                "sbert_score": 1 - results["distances"][0][i],
            }
        )
    return candidates


_ALL_DOCS_CACHE = None


def get_all_documents():
    """Cache seluruh koleksi di memori sekali saja (5000 dokumen, ringan)."""
    global _ALL_DOCS_CACHE
    if _ALL_DOCS_CACHE is None:
        raw = collection.get(include=["metadatas", "documents", "embeddings"])
        _ALL_DOCS_CACHE = list(
            zip(raw["ids"], raw["metadatas"], raw["documents"], raw["embeddings"])
        )
    return _ALL_DOCS_CACHE


def keyword_candidates(
    user_ingredients: list, exclude_ids: set, max_candidates: int = 10
) -> list:
    """Diurutkan berdasarkan (matched_count, coverage_recipe) -- keduanya menurun.
    Tie-breaker coverage_recipe WAJIB ada supaya resep yang lebih efisien tidak
    kepotong hanya karena urutan asli di dataset."""
    scored = []
    for doc_id, meta, document, embedding in get_all_documents():
        if doc_id in exclude_ids:
            continue
        names = json.loads(meta["ingredient_names"])
        matched = sum(
            1 for u in user_ingredients if any(ingredient_matches(u, n) for n in names)
        )
        if matched > 0:
            coverage = matched / len(names) if names else 0.0
            scored.append((matched, coverage, doc_id, meta, document, embedding))

    scored.sort(key=lambda x: (-x[0], -x[1]))
    return [
        (m, doc_id, meta, document, embedding)
        for m, cov, doc_id, meta, document, embedding in scored[:max_candidates]
    ]


def retrieve_candidates_hybrid(
    user_text: str,
    user_ingredients: list,
    top_k_sbert: int = 20,
    max_keyword_extra: int = 10,
) -> list:
    """top_k_sbert=20 + max_keyword_extra=10 -> total maksimal 30 kandidat,
    sesuai proposal final sub-bab 3.1.3.6."""
    query_embedding = model.encode(
        [add_query_prefix(user_text)], normalize_embeddings=True, convert_to_numpy=True
    )[0]
    sbert_candidates = retrieve_candidates(user_text, top_k=top_k_sbert)
    existing_ids = {c["id"] for c in sbert_candidates}
    kw_extra = keyword_candidates(
        user_ingredients, exclude_ids=existing_ids, max_candidates=max_keyword_extra
    )
    for matched, doc_id, meta, document, embedding in kw_extra:
        sbert_score = float(np.dot(query_embedding, np.array(embedding)))
        sbert_candidates.append(
            {
                "id": doc_id,
                "title": meta["title"],
                "ingredient_names": json.loads(meta["ingredient_names"]),
                "sbert_score": sbert_score,
            }
        )
    return sbert_candidates

## 5. Reranking: Weighted Linear Combination

$$skor\_similarity = 1 - distance$$
$$skor\_akhir = w_1 \cdot skor\_similarity + w_2 \cdot skor\_efisiensi\_bahan$$

Bobot final: **w1 = 0,4** dan **w2 = 0,6** (hasil grid search 30 query + validasi kualitatif,
lihat sub-bab 7 dan 8 di bawah).

In [6]:
W1_FINAL = 0.4
W2_FINAL = 0.6


def rerank_hybrid(
    user_text: str,
    w1: float = W1_FINAL,
    w2: float = W2_FINAL,
    top_n: int = 5,
    top_k_sbert: int = 20,
    max_keyword_extra: int = 10,
) -> tuple:
    user_ingredients = parse_user_ingredients(user_text)
    candidates = retrieve_candidates_hybrid(
        user_text, user_ingredients, top_k_sbert, max_keyword_extra
    )

    sbert_scores = np.array([c["sbert_score"] for c in candidates])
    s_min, s_max = sbert_scores.min(), sbert_scores.max()
    sbert_norm = (sbert_scores - s_min) / (s_max - s_min + 1e-9)

    for c, s_norm in zip(candidates, sbert_norm):
        cr, cu, matched, missing = coverage_stats(
            user_ingredients, c["ingredient_names"]
        )
        c["coverage_recipe"] = cr
        c["coverage_user"] = cu
        c["skor_efisiensi_bahan"] = harmonic_efficiency(cr, cu)
        c["sbert_score_norm"] = float(s_norm)
        c["matched_items"] = matched
        c["missing_items"] = missing
        c["final_score"] = w1 * float(s_norm) + w2 * c["skor_efisiensi_bahan"]

    top = sorted(candidates, key=lambda c: c["final_score"], reverse=True)[:top_n]
    return top, user_ingredients

In [7]:
user_input = "2 butir telur, 100gr dada ayam, 2 siung bawang putih"
top, user_ingredients = rerank_hybrid(user_input)

print("Bahan user (setelah parsing):", user_ingredients)
print("=" * 70)
for i, c in enumerate(top):
    print(
        f"[{i + 1}] {c['title']}  (final={c['final_score']:.3f}, sbert_norm={c['sbert_score_norm']:.3f}, "
        f"cov_recipe={c['coverage_recipe']:.3f}, cov_user={c['coverage_user']:.3f}, "
        f"efisiensi={c['skor_efisiensi_bahan']:.3f})"
    )
    print(f"    bahan resep ({len(c['ingredient_names'])}): {c['ingredient_names']}")
    print(f"    -> sudah dimiliki user: {c['matched_items']}")
    print(f"    -> perlu ditambah: {c['missing_items']}")
    print()

Bahan user (setelah parsing): ['telur', 'dada ayam', 'bawang putih']
[1] Ayam kukus telur / steam egg chicken #homemadebylita  (final=0.615, sbert_norm=0.537, cov_recipe=0.500, cov_user=1.000, efisiensi=0.667)
    bahan resep (6): ['dada ayam', 'telur', 'garam', 'merica', 'kaldu jamur', 'bawang putih']
    -> sudah dimiliki user: ['dada ayam', 'telur', 'bawang putih']
    -> perlu ditambah: ['garam', 'merica', 'kaldu jamur']

[2] Telur Dadar  (final=0.609, sbert_norm=0.855, cov_recipe=0.333, cov_user=0.667, efisiensi=0.444)
    bahan resep (6): ['telur', 'bawang merah', 'bawang putih', 'daun bawang', 'garam dan penyedap', 'minyak goreng']
    -> sudah dimiliki user: ['telur', 'bawang putih']
    -> perlu ditambah: ['bawang merah', 'daun bawang', 'garam dan penyedap', 'minyak goreng']

[3] Telur Dadar  (final=0.584, sbert_norm=0.860, cov_recipe=0.286, cov_user=0.667, efisiensi=0.400)
    bahan resep (7): ['telur ayam', 'kornet sapi', 'bawang daun', 'keju cheddar', 'bawang putih bubuk', 

## 6. Guardrail: Cegah Llama3 "Mengarang" Resep

Dua kondisi yang HARUS ditangani sebelum data sampai ke Llama3 -- keduanya rule-based:

1. **Input cuma bumbu/pelengkap, tanpa bahan utama** -- langsung minta bahan utama, tidak usah retrieval.
2. **Kombinasi bahan tidak nyambung** -- retrieval tetap jalan tapi skor terbaiknya sangat rendah;
   kalau di bawah ambang, JANGAN tampilkan sebagai rekomendasi valid.

**Prinsip:** Llama3 HANYA menyampaikan ulang (dalam bahasa natural) resep yang sudah ada di
`results`, atau memparafrase kalimat template untuk kondisi gagal -- tidak pernah membuat
konten resep sendiri.

In [8]:
import re  # dibutuhkan is_main_ingredient() untuk word-boundary matching

# Daftar kata kunci bahan utama, disusun berdasarkan 7 dari 13 kelompok pangan resmi
# panganku.org (lihat Lampiran 1 proposal). Setiap kata kunci disusun secara manual
# dengan memeriksa entri satu per satu (bukan ekstraksi otomatis), termasuk penyertaan
# bagian organ secara terpisah dari nama hewannya (mis. "hati" terpisah dari "sapi").
MAIN_INGREDIENT_KEYWORDS = {
    "Serealia": {
        "beras",
        "nasi",
        "ketan",
        "jagung",
        "cantel",
        "jali",
        "jawawut",
        "jampang huma",
        "tepung beras",
        "tepung jagung",
        "tepung terigu",
        "maizena",
        "bihun",
        "mie",
        "mi",
        "misoa",
        "makaroni",
        "spaghetti",
        "roti",
    },
    "Umbi Berpati": {
        "kentang",
        "singkong",
        "ketela pohon",
        "ubi",
        "ubi jalar",
        "ubi cilembu",
        "talas",
        "gembili",
        "gadung",
        "ganyong",
        "suweg",
        "uwi",
        "sente",
        "bengkuang",
        "arrowroot",
        "sagu",
        "tapioka",
        "tepung singkong",
        "tepung kentang",
        "tepung ganyong",
        "tepung mocaf",
        "mocaf",
        "tepung arrowroot",
        "tepung ubi",
    },
    "Kacang-kacangan": {
        "kacang arab",
        "kacang bogor",
        "kacang ercis",
        "kacang gude",
        "kacang hijau",
        "kacang hitam",
        "kacang kapri",
        "kedelai",
        "kacang kedelai",
        "kacang komak",
        "kacang merah",
        "kacang mete",
        "kacang tanah",
        "kacang tolo",
        "kacang tunggak",
        "kacang panjang",
        "kenari",
        "koro",
        "lamtoro",
        "wijen",
        "tahu",
        "tempe",
        "oncom",
        "kembang tahu",
        "susu kedelai",
        "tauco",
        "tepung hunkwe",
        "hunkwe",
        "tepung jalejo",
        "tepung kacang kedelai",
        "tepung kedelai",
    },
    "Sayuran": {
        "akar tonjong",
        "bit",
        "lobak",
        "rebung",
        "umbut kelapa",
        "umbut rotan",
        "ale",
        "toge",
        "taoge",
        "taoge kacang kedelai",
        "taoge kacang tunggak",
        "andewi",
        "bayam",
        "bayam merah",
        "daun bangun-bangun",
        "daun bebuas",
        "daun belem",
        "daun bluntas",
        "daun gandaria",
        "daun gedi",
        "daun gedi besar",
        "daun gedi kecil",
        "daun gelang",
        "daun gunda",
        "daun gunda bali",
        "daun gunda serang",
        "daun jambu mete",
        "daun jampang",
        "daun jawaw seluang",
        "daun jonghe",
        "daun kacang ma",
        "daun kacang panjang",
        "daun kasbi",
        "daun singkong karet",
        "daun katuk",
        "daun kecipir",
        "daun kedondong",
        "daun kelor",
        "daun kemang",
        "daun kenikir",
        "kenikir",
        "daun kol sawi",
        "daun koro",
        "daun kubis",
        "daun kumak",
        "daun labu siam",
        "daun labu",
        "daun labu waluh",
        "daun labu kuning",
        "daun lamtoro",
        "daun leilem",
        "daun leunca",
        "daun lobak",
        "daun lompong",
        "daun lompong talas",
        "daun mangkokan",
        "daun matel ambon",
        "daun melinjo",
        "daun mengkudu",
        "daun ndusuk",
        "daun oyong",
        "daun pakis",
        "daun pakis longgiho",
        "daun pakis wambateu",
        "daun paku",
        "daun pangi",
        "daun pare",
        "daun pepaya",
        "daun pohpohan",
        "pohpohan",
        "daun semanggi",
        "semanggi",
        "daun simpur",
        "daun singkil",
        "daun singkong",
        "daun singkong ambon",
        "daun singkong ampenan",
        "daun singkong kopang",
        "daun sintrong",
        "sintrong",
        "daun talas",
        "daun tespong",
        "tespong",
        "daun ubi",
        "daun ubi kuning",
        "daun ubi merah",
        "daun ubi putih",
        "daun ubi tinta",
        "eceng",
        "genjer",
        "kalakai",
        "kangkung",
        "kangkung tondano",
        "selada",
        "selada air",
        "sawi",
        "sawi putih",
        "pecai",
        "sawi taiwan",
        "sawi tanah",
        "mostarda metan",
        "baligo",
        "buah kelor",
        "buah merah",
        "gambas",
        "oyong",
        "jagung muda",
        "jagung semi",
        "baby corn",
        "jantung pisang",
        "jotang",
        "kabau",
        "karawila",
        "kelawi",
        "kluwih",
        "ketimun",
        "timun",
        "ketimun krai",
        "ketimun madura",
        "kundur",
        "labu air",
        "labu kuning",
        "labu siam",
        "labu waluh",
        "waluh",
        "lantar",
        "leunca",
        "lumai",
        "nangka muda",
        "paria putih",
        "pare",
        "paria",
        "pepare ular",
        "pepaya muda",
        "rimbang",
        "tekokak",
        "terong",
        "terung",
        "terong asam",
        "terung belanda",
        "terung bengkulu",
        "terong hintalo",
        "terung panjang",
        "tomat",
        "tomat merah",
        "tomat muda",
        "buncis",
        "kacang mekah",
        "kacang panjang",
        "kacang ranti",
        "kapri muda",
        "kecipir muda",
        "komak",
        "koro kerupuk",
        "koro wedus",
        "caisin",
        "kool kembang",
        "kembang kol",
        "kol kembang",
        "kool merah",
        "kol merah",
        "kool putih",
        "kol putih",
        "kubis",
        "kol",
        "bunga pepaya",
        "bunga turi",
        "kembang turi",
        "uceng",
        "bunga melinjo",
        "jamur",
        "jamur encik",
        "jamur kuping",
        "jamur merang",
        "jamur tiram",
        "jamur sagu",
        "jengkol",
        "kerokot",
        "kulit melinjo",
        "melinjo",
        "petai",
        "putri malu",
        "rumput laut",
        "tebu terubuk",
        "terubuk",
        "wortel",
    },
    "Daging": {
        "sapi",
        "ayam",
        "angsa",
        "babi",
        "bebek",
        "itik",
        "belibis",
        "burung",
        "domba",
        "kambing",
        "kelinci",
        "kerbau",
        "kuda",
        "ulat sagu",
        "buaya",
        "menjangan",
        "punai",
        "rusa",
        "penyu",
        "anjing",
        "tikus",
        "daging",
        "babat",
        "jeroan",
        "daleman",
        "hati",
        "ginjal",
        "usus",
        "otak",
        "kikil",
        "buntut",
        "ampela",
        "ham",
        "dendeng",
        "kornet",
        "sosis",
    },
    "Ikan/Kerang/Udang/dll": {
        "belut",
        "belut laut",
        "cumi-cumi",
        "cumi",
        "ikan",
        "ikan bader",
        "ikan balong",
        "ikan bambangan",
        "ikan bandeng",
        "bandeng",
        "ikan banjar",
        "ikan baronang",
        "baronang",
        "ikan batung",
        "ikan baung",
        "baung",
        "ikan bawal",
        "bawal",
        "ikan belida",
        "belida",
        "ikan beunteur",
        "ikan biawan",
        "ikan bili",
        "ikan bubara",
        "ikan bulan-bulan",
        "ikan cakalang",
        "cakalang",
        "ikan daun",
        "ikan ekor kuning",
        "ekor kuning",
        "ikan gabus",
        "gabus",
        "ikan heu",
        "ikan hitam",
        "ikan hiu",
        "hiu",
        "ikan kacangan",
        "ikan kakap",
        "kakap",
        "kakap merah",
        "ikan kakatua",
        "ikan kalaban",
        "ikan kamera",
        "ikan kapar",
        "ikan kawalinya",
        "ikan keru-keru",
        "ikan kima",
        "ikan lais",
        "lais",
        "ikan layang",
        "layang",
        "ikan layur",
        "layur",
        "ikan lehoma",
        "ikan lemuru",
        "lemuru",
        "ikan lidah",
        "ikan malalugis",
        "ikan mamar merah",
        "ikan mas",
        "ikan mayong",
        "ikan mujahir",
        "mujahir",
        "ikan nasu metti",
        "ikan oci",
        "ikan kembung",
        "kembung",
        "ikan paling",
        "ikan papuyu",
        "papuyu",
        "betok",
        "ikan patin",
        "patin",
        "ikan pomo",
        "ikan puntin",
        "ikan saluang",
        "saluang",
        "ikan sarden",
        "sarden",
        "ikan selar",
        "selar",
        "ikan sepat",
        "sepat",
        "ikan sidat",
        "sidat",
        "ikan sunu",
        "sunu",
        "ikan tahuman",
        "ikan tarmon",
        "ikan telan",
        "ikan tembang",
        "ikan tempahas",
        "ikan terbang",
        "ikan teri",
        "teri",
        "ikan titang",
        "ikan tongkol",
        "tongkol",
        "ikan turi",
        "keong",
        "kepiting",
        "kerang",
        "kodok",
        "kura-kura",
        "kuro",
        "lokan",
        "rajungan",
        "rebon",
        "udang",
        "udang galah",
        "ikan asin",
        "ikan kering",
        "ikan asap",
        "ikan pindang",
        "pindang",
        "ikan sale",
        "ikan kayu",
        "peda",
        "jambal",
        "sarden kaleng",
        "sardines dalam kaleng",
        "rebon kering",
        "udang kering",
        "abon ikan",
        "tepung ikan",
        "tepung teri",
    },
    "Telur": {
        "telur",
    },
}

# gabungan seluruh kata kunci jadi satu set datar, dipakai untuk pengecekan cepat
MAIN_INGREDIENT_KEYWORDS_FLAT = set().union(*MAIN_INGREDIENT_KEYWORDS.values())


def is_main_ingredient(name: str) -> bool:
    """Cek apakah nama bahan cocok dengan salah satu kata kunci bahan utama pada
    Lampiran 1. Kata kunci dengan panjang <=5 huruf (mis. "mi", "kol", "teri", "ubi")
    dicocokkan menggunakan word-boundary (harus jadi kata utuh), BUKAN substring bebas --
    soalnya substring bebas pada kata kunci pendek terbukti (lewat testing) bisa salah
    kena kata yang tidak berkaitan sama sekali (mis. "mi"/"teri" nyempil di "misterius").
    Kata kunci yang lebih panjang (>5 huruf) tetap memakai pencocokan substring dua arah
    biasa (ingredient_matches), supaya kombinasi seperti "dada ayam" tetap cocok dengan
    kata kunci "ayam"."""
    name = name.lower().strip()
    for kw in MAIN_INGREDIENT_KEYWORDS_FLAT:
        if len(kw) <= 5:
            if re.search(r"\b" + re.escape(kw) + r"\b", name):
                return True
        else:
            if ingredient_matches(name, kw):
                return True
    return False


def classify_ingredient(name: str) -> str:
    """'bahan_utama' kalau cocok dengan salah satu kata kunci Lampiran 1,
    selain itu 'belum_terklasifikasi' (BUKAN otomatis 'bumbu' -- lihat catatan
    is_main_ingredient di atas)."""
    return "bahan_utama" if is_main_ingredient(name) else "belum_terklasifikasi"


def has_main_ingredient(user_ingredients: list) -> bool:
    """T_kategori: True kalau minimal 1 bahan pengguna cocok dengan salah satu
    kata kunci bahan utama pada Lampiran 1 (pengecekan positif/inclusion,
    bukan exclusion terhadap daftar bumbu)."""
    return any(is_main_ingredient(u) for u in user_ingredients)


# test -- kasus yang sama seperti sebelumnya, plus kasus baru untuk cek "belum terklasifikasi"
# dan kasus false-positive kata kunci pendek yang ditemukan lewat testing
print(
    has_main_ingredient(["garam", "gula", "kecap manis"])
)  # False -> tidak ada yg cocok kata kunci manapun
print(has_main_ingredient(["telur", "garam"]))  # True  -> "telur" cocok kelompok Telur
print(has_main_ingredient(["tahu", "tempe"]))  # True  -> cocok kelompok Kacang-kacangan
print(has_main_ingredient(["hati sapi"]))  # True  -> cocok "hati" & "sapi" (Daging)
print(
    has_main_ingredient(["bahan misterius xyz"])
)  # False -> belum terklasifikasi, bukan bumbu

False
True
True
True
False


In [9]:
def is_relevant_enough(best_candidate: dict, user_ingredients: list) -> bool:
    """T_relevansi = min(2, jumlah bahan user):
    - user kasih 1 bahan -> wajib match bahan itu (T=1)
    - user kasih >=2 bahan -> wajib match minimal 2 bahan (T=2)
    Threshold tetap (mis. selalu >=2) salah menolak query pendek yang valid; threshold
    longgar (>=1) meloloskan kombinasi yang sebenarnya tidak related (lihat kasus
    'soda kue, kunyit' vs 'ayam, gula')."""
    if not best_candidate or not user_ingredients:
        return False
    T_relevansi = min(2, len(user_ingredients))
    return len(best_candidate.get("matched_items", [])) >= T_relevansi


def handle_user_input(user_text: str, top_n: int = 5) -> dict:
    """Pipeline utuh dgn guardrail. Return dict dgn 'status' yg menentukan respons apa
    yang boleh disampaikan Llama3 -- Llama3 TIDAK pernah membuat konten resep sendiri."""
    user_ingredients = parse_user_ingredients(user_text)

    if not user_ingredients:
        return {
            "status": "no_ingredients",
            "template": "Sepertinya belum ada bahan yang terdeteksi dari pesanmu. Boleh sebutkan bahan yang kamu punya?",
        }

    if not has_main_ingredient(user_ingredients):
        return {
            "status": "bumbu_only",
            "user_ingredients": user_ingredients,
            "template": (
                f"Bahan yang kamu sebutkan ({', '.join(user_ingredients)}) sepertinya baru bumbu/pelengkap. "
                f"Ada bahan utama juga nggak, misalnya ayam, telur, tahu, atau sayuran?"
            ),
        }

    results, _ = rerank_hybrid(user_text, top_n=top_n)

    if not is_relevant_enough(results[0] if results else None, user_ingredients):
        return {
            "status": "no_match",
            "user_ingredients": user_ingredients,
            "template": (
                f"Maaf, sistem belum menemukan resep yang cukup cocok untuk kombinasi bahan "
                f"({', '.join(user_ingredients)}). Coba sebutkan bahan lain atau kombinasi berbeda ya."
            ),
        }

    return {
        "status": "ok",
        "user_ingredients": user_ingredients,
        "results": results,
    }


# test ketiga kasus
for test_input in [
    "garam, gula, kecap manis",
    "soda kue, kunyit",
    "2 butir telur, 100gr dada ayam, 2 siung bawang putih",
]:
    print(f"Input: {test_input!r}")
    result = handle_user_input(test_input)
    print(f"  status: {result['status']}")
    if result["status"] != "ok":
        print(f"  template: {result['template']}")
    else:
        print(f"  top-1: {result['results'][0]['title']}")
    print()

Input: 'garam, gula, kecap manis'
  status: bumbu_only
  template: Bahan yang kamu sebutkan (garam, gula, kecap manis) sepertinya baru bumbu/pelengkap. Ada bahan utama juga nggak, misalnya ayam, telur, tahu, atau sayuran?

Input: 'soda kue, kunyit'
  status: bumbu_only
  template: Bahan yang kamu sebutkan (soda kue, kunyit) sepertinya baru bumbu/pelengkap. Ada bahan utama juga nggak, misalnya ayam, telur, tahu, atau sayuran?

Input: '2 butir telur, 100gr dada ayam, 2 siung bawang putih'
  status: ok
  top-1: Ayam kukus telur / steam egg chicken #homemadebylita



## 7. Grid Search Bobot (w1, w2)

Menjalankan grid search menggunakan 30 query uji (sama dengan yang dilaporkan di BAB I) untuk
menentukan w1 dan w2. Bobot dipilih berdasarkan **kombinasi** trade-off mean_IAR@5 vs
mean_SBERT@5 (elbow point) DAN validasi kualitatif (sub-bab 8) -- metrik agregat saja
terbukti bisa menyarankan bobot yang secara kualitatif kurang baik (lihat catatan di akhir
sub-bab ini).

In [10]:
test_queries_full = [
    "2 butir telur, 100gr dada ayam, 2 siung bawang putih",
    "200gr daging sapi, kecap manis, bawang bombay",
    "tahu, tempe, cabai rawit",
    "udang, tepung terigu, telur",
    "wortel, kentang, buncis",
    "tempe, kacang panjang, cabai merah",
    "2 butir telur",
    "nasi sisa kemarin, 1 butir telur",
    "1 papan tempe, kecap manis",
    "dada ayam fillet dikit",
    "indomie goreng, sawi hijau, cabe rawit",
    "roti tawar, keju cheddar",
    "1 buah tahu, 1 buah tomat, daun bawang",
    "500 gr daging sapi, 1 buah bawang bombay, kecap inggris",
    "1 ikat bayam, jagung manis, bawang merah",
    "kangkung, terasi, cabai rawit, bawang merah",
    "ikan nila, bumbu kuning, kemangi",
    "sosis sapi, nasi putih, bawang bombay, saus sambal",
    "200 gr udang, brokoli, bawang putih, saus tiram",
    "ikan lele, jeruk nipis, garam, ketumbar",
    "500gr ayam, santan kental, daun salam, serai, lengkuas, bawang merah, bawang putih",
    "daging sapi, kelapa sangrai, cabai merah besar, bawang merah, bawang putih, jahe, lengkuas",
    "nasi putih, telur, sosis, bakso, kecap manis, saus tomat, sayur kol, daun bawang",
    "dada ayam, tepung bumbu, telur, bawang putih, saus asam manis, nanas, paprika",
    "tahu, tempe, kacang panjang, tauge, bumbu kacang, kerupuk",
    "tepung terigu, mentega, gula pasir, telur, vanili",
    "pisang kepok, tepung beras, gula merah, minyak goreng",
    "bubuk matcha murni, susu cair, es batu",
    "250 gr jamur enoki, gochujang, minyak wijen",
    "frozen kentang goreng, sosis sapi, bumbu tabur jagung bakar",
]
print(f"Jumlah query uji: {len(test_queries_full)}")

Jumlah query uji: 30


In [11]:
def evaluate_weight(w1: float, w2: float, top_n: int = 5) -> dict:
    """Rata-rata IAR@top_n (=coverage_recipe) dan SBERT@top_n untuk kombinasi bobot tertentu.
    PENTING: skor_efisiensi_bahan pakai harmonic_efficiency (coverage_recipe, coverage_user),
    bukan coverage_recipe saja -- lihat sub-bab 3."""
    iars, sberts = [], []
    for q in test_queries_full:
        user_ingredients = parse_user_ingredients(q)
        candidates = retrieve_candidates_hybrid(
            q, user_ingredients, top_k_sbert=20, max_keyword_extra=10
        )
        sbert_scores = np.array([c["sbert_score"] for c in candidates])
        s_min, s_max = sbert_scores.min(), sbert_scores.max()
        sbert_norm = (sbert_scores - s_min) / (s_max - s_min + 1e-9)
        for c, s_norm in zip(candidates, sbert_norm):
            cr, cu, matched, missing = coverage_stats(
                user_ingredients, c["ingredient_names"]
            )
            c["coverage_recipe"] = cr
            c["skor_efisiensi_bahan"] = harmonic_efficiency(cr, cu)
            c["sbert_score_norm"] = float(s_norm)
            c["final_score"] = w1 * float(s_norm) + w2 * c["skor_efisiensi_bahan"]
        top = sorted(candidates, key=lambda c: c["final_score"], reverse=True)[:top_n]
        iars.append(np.mean([c["coverage_recipe"] for c in top]))
        sberts.append(np.mean([c["sbert_score_norm"] for c in top]))
    return {"mean_iar": float(np.mean(iars)), "mean_sbert": float(np.mean(sberts))}


print(f"{'w1':>6} {'w2':>6} {'mean_CoverageRecipe@5':>12} {'mean_SBERT@5':>14}")
results_grid = []
for w1 in np.arange(0.1, 1.0, 0.1):
    w2 = round(1 - w1, 1)
    w1 = round(w1, 1)
    metrics = evaluate_weight(w1, w2)
    results_grid.append((w1, w2, metrics["mean_iar"], metrics["mean_sbert"]))
    print(
        f"{w1:>6} {w2:>6} {metrics['mean_iar']:>12.4f} {metrics['mean_sbert']:>14.4f}"
    )

print(
    "\nCatatan: setelah retrieval diubah menjadi top_k_sbert=20 + max_keyword_extra=10"
)
print(
    "(total maksimal 30 kandidat, sesuai proposal final 3.1.3.6), bandingkan tabel di atas"
)
print(
    "dengan Tabel 3.2 pada proposal (yang dihasilkan dari pengaturan retrieval lama, 30+20=50)."
)
print(
    "Kalau pola dan kesimpulan (elbow point sekitar w1=0.6, w1=0.4 lebih aman secara kualitatif)"
)
print(
    "masih konsisten, cukup dicatat sebagai verifikasi ulang. Kalau berbeda signifikan,"
)
print(
    "Tabel 3.2 dan Lampiran terkait grid search perlu diperbarui dengan angka baru ini."
)

    w1     w2 mean_CoverageRecipe@5   mean_SBERT@5
   0.1    0.9       0.3645         0.6268
   0.2    0.8       0.3535         0.7250
   0.3    0.7       0.3389         0.7873
   0.4    0.6       0.3200         0.8244
   0.5    0.5       0.3039         0.8502
   0.6    0.4       0.2780         0.8775
   0.7    0.3       0.2526         0.8917
   0.8    0.2       0.2327         0.8993
   0.9    0.1       0.2219         0.9022

Catatan: setelah retrieval diubah menjadi top_k_sbert=20 + max_keyword_extra=10
(total maksimal 30 kandidat, sesuai proposal final 3.1.3.6), bandingkan tabel di atas
dengan Tabel 3.2 pada proposal (yang dihasilkan dari pengaturan retrieval lama, 30+20=50).
Kalau pola dan kesimpulan (elbow point sekitar w1=0.6, w1=0.4 lebih aman secara kualitatif)
masih konsisten, cukup dicatat sebagai verifikasi ulang. Kalau berbeda signifikan,
Tabel 3.2 dan Lampiran terkait grid search perlu diperbarui dengan angka baru ini.


## 8. Spot-Check Kualitatif dengan Bobot Final

Validasi manual: lihat langsung top-5 hasil di beberapa query pakai bobot final. Tujuannya
mastiin hasilnya masuk akal secara kualitatif, bukan cuma bagus di angka agregat -- ingat
temuan di sub-bab 7 bahwa metrik agregat saja bisa menyarankan bobot yang buruk secara kualitatif.

In [12]:
spot_check_queries = [
    "2 butir telur, 100gr dada ayam, 2 siung bawang putih",
    "tahu, tempe, cabai rawit",
    "ikan lele, jeruk nipis, garam, ketumbar",
    "daging sapi, kecap manis, bawang bombay",
]

for q in spot_check_queries:
    top, user_ingredients = rerank_hybrid(q, w1=W1_FINAL, w2=W2_FINAL)
    print(f"Query: {q!r}")
    print(f"Bahan user (setelah parsing): {user_ingredients}")
    print("-" * 70)
    for i, c in enumerate(top):
        print(
            f"[{i + 1}] {c['title']}  (final={c['final_score']:.3f}, sbert_norm={c['sbert_score_norm']:.3f}, "
            f"cov_recipe={c['coverage_recipe']:.3f}, cov_user={c['coverage_user']:.3f}, "
            f"efisiensi={c['skor_efisiensi_bahan']:.3f})"
        )
        print(
            f"    bahan resep ({len(c['ingredient_names'])}): {c['ingredient_names']}"
        )
        print(f"    -> sudah dimiliki user: {c['matched_items']}")
        print(f"    -> perlu ditambah: {c['missing_items']}")
    print("=" * 70)
    print()

print("CHECKLIST VALIDASI MANUAL:")
print("[ ] Apakah top-1 di tiap query terlihat masuk akal secara tema/topik masakan?")
print(
    "[ ] Apakah 'bahan yang perlu ditambah' terlihat wajar (bumbu dasar), bukan bahan aneh?"
)
print("[ ] Apakah resep dgn cov_user rendah (cuma nyerempet 1 bahan) TIDAK menang lagi")
print("    dibanding resep dengan cov_user tinggi (pakai semua bahan user)?")
print("Kalau ada hasil yang janggal, catat sebagai limitasi di BAB IV.")

Query: '2 butir telur, 100gr dada ayam, 2 siung bawang putih'
Bahan user (setelah parsing): ['telur', 'dada ayam', 'bawang putih']
----------------------------------------------------------------------
[1] Ayam kukus telur / steam egg chicken #homemadebylita  (final=0.615, sbert_norm=0.537, cov_recipe=0.500, cov_user=1.000, efisiensi=0.667)
    bahan resep (6): ['dada ayam', 'telur', 'garam', 'merica', 'kaldu jamur', 'bawang putih']
    -> sudah dimiliki user: ['dada ayam', 'telur', 'bawang putih']
    -> perlu ditambah: ['garam', 'merica', 'kaldu jamur']
[2] Telur Dadar  (final=0.609, sbert_norm=0.855, cov_recipe=0.333, cov_user=0.667, efisiensi=0.444)
    bahan resep (6): ['telur', 'bawang merah', 'bawang putih', 'daun bawang', 'garam dan penyedap', 'minyak goreng']
    -> sudah dimiliki user: ['telur', 'bawang putih']
    -> perlu ditambah: ['bawang merah', 'daun bawang', 'garam dan penyedap', 'minyak goreng']
[3] Telur Dadar  (final=0.584, sbert_norm=0.860, cov_recipe=0.286, cov_us